# Step 2 — Manual ROI Review

- Left: 4×4 grid of DFF traces + numbered buttons to select a ROI
- Right: FOV with selected ROI highlighted + full DFF trace
- **✓ Good** / **✗ Bad** buttons label the selected ROI and auto-advance
- Labels auto-save to `ROI_label.h5` after every action


In [1]:
%matplotlib inline

import os, sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.ndimage import gaussian_filter

## Configuration — edit these values

## Session loading helpers

In [2]:
def _session_path(date):
    prefix = MOUSE_FOLDER.split('_')[0]
    return os.path.join(BASE_PATH, MOUSE_FOLDER, f'{prefix}_{date}')


def discover_sessions():
    if DATES:
        paths = [_session_path(d) for d in DATES]
    else:
        mouse_dir = os.path.join(BASE_PATH, MOUSE_FOLDER)
        prefix    = MOUSE_FOLDER.split('_')[0]
        paths = sorted([
            os.path.join(mouse_dir, d)
            for d in os.listdir(mouse_dir)
            if d.startswith(prefix)
            and os.path.isdir(os.path.join(mouse_dir, d, 'qc_results'))
        ])
    valid = [p for p in paths if os.path.isdir(os.path.join(p, 'qc_results'))]
    missing = [p for p in paths if p not in valid]
    if missing:
        print('WARNING — qc_results not found (run step 1 first):')
        for p in missing:
            print(f'  {p}')
    return valid


def load_session(path):
    ops = np.load(
        os.path.join(path, 'suite2p', 'plane0', 'ops.npy'),
        allow_pickle=True).item()
    ops['save_path0'] = path

    qc       = os.path.join(path, 'qc_results')
    fluo     = np.load(os.path.join(qc, 'fluo.npy'),     allow_pickle=True)
    neuropil = np.load(os.path.join(qc, 'neuropil.npy'), allow_pickle=True)
    masks    = np.load(os.path.join(qc, 'masks.npy'),    allow_pickle=True)

    neucoeff     = float(ops.get('neucoeff', 0.7))
    sig_baseline = int(ops.get('sig_baseline', config.SIG_BASELINE))
    dff = fluo.astype(np.float64) - neucoeff * neuropil.astype(np.float64)
    f0  = gaussian_filter(dff, [0., sig_baseline])
    dff = (dff - f0) / (f0 + 1e-10)

    n_rois = dff.shape[0]
    labels = np.zeros(n_rois, dtype=np.int8)
    label_path = os.path.join(path, 'ROI_label.h5')
    if os.path.isfile(label_path):
        with h5py.File(label_path, 'r') as fh:
            good = np.array(fh['good_roi'], dtype=int)
            bad  = np.array(fh['bad_roi'],  dtype=int)
        good = good[good < n_rois]
        bad  = bad[bad   < n_rois]
        labels[good] = 1
        labels[bad]  = -1
        print(f'  Loaded existing labels: {len(good)} good, {len(bad)} bad')

    x1, x2 = ops['xrange'][0], ops['xrange'][1]
    y1, y2 = ops['yrange'][0], ops['yrange'][1]
    mean_img   = ops['meanImg'][y1:y2, x1:x2]
    masks_crop = masks[y1:y2, x1:x2]

    return {
        'path':     path,
        'dff':      dff,
        'masks':    masks_crop,
        'mean_img': mean_img,
        'labels':   labels,
        'n_rois':   n_rois,
    }


def save_labels(sess):
    labels   = sess['labels']
    good_roi = np.where(labels == 1)[0].astype(np.int64)
    bad_roi  = np.where(labels == -1)[0].astype(np.int64)
    with h5py.File(os.path.join(sess['path'], 'ROI_label.h5'), 'w') as fh:
        fh['good_roi'] = good_roi
        fh['bad_roi']  = bad_roi

In [3]:
class ROIReviewer:
    PAGE_SIZE = 16
    ROWS = 4
    COLS = 4
    # label → ipywidgets button_style
    BTN_STYLE = {1: 'success', -1: 'danger', 0: ''}

    def __init__(self, session_paths):
        self._paths = session_paths
        self._si    = 0
        self._page  = 0
        self._sel   = 0
        self._sess  = None
        self._load_session(0)
        self._build_ui()

    # ── data ──────────────────────────────────────────────────────────────────

    def _load_session(self, idx):
        self._si   = idx
        self._page = 0
        self._sel  = 0
        print(f'Loading session {idx+1}/{len(self._paths)}: '
              f'{os.path.basename(self._paths[idx])} …')
        self._sess = load_session(self._paths[idx])
        print(f'  {self._sess["n_rois"]} ROIs  |  {self._n_pages} pages')

    @property
    def _n_pages(self):
        return max(1, (self._sess['n_rois'] + self.PAGE_SIZE - 1) // self.PAGE_SIZE)

    @property
    def _page_rois(self):
        start = self._page * self.PAGE_SIZE
        return list(range(start, min(start + self.PAGE_SIZE, self._sess['n_rois'])))

    @property
    def _selected_roi(self):
        rois = self._page_rois
        return rois[min(self._sel, len(rois) - 1)]

    def _mid_window(self):
        """Return (start, end) frame indices for a 30 s window centred on the session."""
        n   = self._sess['dff'].shape[1]
        win = int(DISPLAY_SECONDS * config.FS)
        mid = n // 2
        start = max(0, mid - win // 2)
        end   = min(n, start + win)
        return start, end

    # ── UI construction ───────────────────────────────────────────────────────

    def _build_ui(self):
        # output areas for matplotlib figures
        self._grid_out   = widgets.Output()
        self._detail_out = widgets.Output()

        # 4x4 grid of selection buttons
        self._gbtn = [
            widgets.Button(
                description='',
                layout=widgets.Layout(width='78px', height='26px'),
            )
            for _ in range(self.PAGE_SIZE)
        ]
        for i, btn in enumerate(self._gbtn):
            btn._ci = i
            btn.on_click(self._on_grid_btn)

        grid_btn_box = widgets.GridBox(
            self._gbtn,
            layout=widgets.Layout(
                grid_template_columns='repeat(4, 82px)',
                width='340px',
            )
        )

        # action buttons
        bl   = widgets.Layout(width='95px',  height='32px')
        bl_w = widgets.Layout(width='115px', height='32px')

        self._btn_good = widgets.Button(description='✓ Good (G)', button_style='success', layout=bl)
        self._btn_bad  = widgets.Button(description='✗ Bad (B)',  button_style='danger',  layout=bl)
        self._btn_skip = widgets.Button(description='→ Skip',                             layout=bl)
        self._btn_pp   = widgets.Button(description='◄ Page',                             layout=bl)
        self._btn_np   = widgets.Button(description='Page ►',                             layout=bl)
        self._btn_ps   = widgets.Button(description='◄ Session',                          layout=bl_w)
        self._btn_ns   = widgets.Button(description='Session ►',                          layout=bl_w)
        self._status   = widgets.HTML(value='')

        self._btn_good.on_click(lambda _: self._label(1))
        self._btn_bad.on_click( lambda _: self._label(-1))
        self._btn_skip.on_click(lambda _: self._advance())
        self._btn_pp.on_click(  lambda _: self._prev_page())
        self._btn_np.on_click(  lambda _: self._next_page())
        self._btn_ps.on_click(  lambda _: self._prev_session())
        self._btn_ns.on_click(  lambda _: self._next_session())

        action_row = widgets.HBox([
            self._btn_ps, self._btn_pp,
            widgets.Label('  '),
            self._btn_good, self._btn_bad, self._btn_skip,
            widgets.Label('  '),
            self._btn_np, self._btn_ns,
        ])

        left_panel  = widgets.VBox([self._grid_out, grid_btn_box])
        main_panels = widgets.HBox([left_panel, self._detail_out])

        display(self._status)
        display(main_panels)
        display(action_row)

        self._draw_grid()
        self._draw_detail()
        self._update_grid_btns()
        self._update_status()

    # ── drawing ───────────────────────────────────────────────────────────────

    def _draw_grid(self):
        dff    = self._sess['dff']
        labels = self._sess['labels']
        rois   = self._page_rois
        f0, f1 = self._mid_window()
        t      = np.arange(f1 - f0) / config.FS

        with self._grid_out:
            clear_output(wait=True)
            fig, axes = plt.subplots(self.ROWS, self.COLS, figsize=(13, 7))
            fig.subplots_adjust(hspace=0.4, wspace=0.15,
                                left=0.02, right=0.98, top=0.95, bottom=0.04)

            trace_color = {1: '#27ae60', -1: '#e74c3c', 0: '#2c3e50'}
            bg_color    = {1: '#eafaf1', -1: '#fdedec', 0: '#f8f9fa'}

            for ci, ax in enumerate(axes.flat):
                ax.set_xticks([]); ax.set_yticks([])
                if ci >= len(rois):
                    ax.set_visible(False)
                    continue

                roi    = rois[ci]
                lbl    = int(labels[roi])
                is_sel = (ci == self._sel)

                ax.set_facecolor(bg_color.get(lbl, '#f8f9fa'))
                edge_c = '#f39c12' if is_sel else trace_color.get(lbl, '#7f8c8d')

                # only left and bottom spines — coloured to show label status
                for name, sp in ax.spines.items():
                    if name in ('top', 'right'):
                        sp.set_visible(False)
                    else:
                        sp.set_edgecolor(edge_c)
                        sp.set_linewidth(2.5 if is_sel else 0.8)

                ax.plot(t, dff[roi, f0:f1], lw=0.5,
                        color=trace_color.get(lbl, '#2c3e50'))
                ax.autoscale(axis='y', tight=True)
                ax.set_title(f'#{roi}', fontsize=6, pad=1, color=edge_c,
                             fontweight='bold' if is_sel else 'normal')

            plt.show()

    def _draw_detail(self):
        sel  = self._selected_roi
        dff  = self._sess['dff']
        mask = self._sess['masks']
        img  = self._sess['mean_img']
        f0, f1 = self._mid_window()
        t    = np.arange(f1 - f0) / config.FS

        with self._detail_out:
            clear_output(wait=True)
            fig, (ax_fov, ax_tr) = plt.subplots(2, 1, figsize=(4, 7))
            fig.subplots_adjust(hspace=0.35,
                                left=0.1, right=0.95, top=0.93, bottom=0.08)

            ax_fov.imshow(img, cmap='gray', aspect='equal', interpolation='nearest')
            overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
            overlay[mask == sel + 1] = [1.0, 0.55, 0.0, 0.8]
            ax_fov.imshow(overlay, aspect='equal', interpolation='nearest')
            ax_fov.set_title(f'ROI #{sel}', fontsize=9)
            ax_fov.axis('off')

            t_full = np.arange(dff.shape[1]) / config.FS
            ax_tr.plot(t_full, dff[sel], lw=0.75, color='#2c3e50')
            ax_tr.set_xlabel('Time (s)', fontsize=8)
            ax_tr.set_ylabel('ΔF/F',     fontsize=8)
            ax_tr.spines[['top', 'right']].set_visible(False)
            ax_tr.tick_params(labelsize=7)
            ax_tr.set_title(f'Full trace — ROI #{sel}', fontsize=8)


            plt.show()

    def _update_grid_btns(self):
        rois   = self._page_rois
        labels = self._sess['labels']
        for ci, btn in enumerate(self._gbtn):
            if ci >= len(rois):
                btn.description  = ''
                btn.button_style = ''
                btn.disabled     = True
            else:
                roi    = rois[ci]
                lbl    = int(labels[roi])
                is_sel = (ci == self._sel)
                btn.description  = f'#{roi}'
                btn.disabled     = False
                btn.button_style = 'warning' if is_sel else self.BTN_STYLE[lbl]

    def _update_status(self):
        lbs  = self._sess['labels']
        n    = self._sess['n_rois']
        good = int((lbs == 1).sum())
        bad  = int((lbs == -1).sum())
        pend = int((lbs == 0).sum())
        name = os.path.basename(self._sess['path'])
        self._status.value = (
            f'<b>Session {self._si+1}/{len(self._paths)}: {name}</b>  │  '
            f'Page {self._page+1}/{self._n_pages}  │  '
            f'<span style="color:#27ae60">✓ {good}</span>  '
            f'<span style="color:#e74c3c">✗ {bad}</span>  '
            f'<span style="color:#7f8c8d">? {pend}</span>'
            f' / {n}  │  Selected: <b>#{self._selected_roi}</b>'
        )

    # ── event handlers ────────────────────────────────────────────────────────

    def _on_grid_btn(self, btn):
        ci = btn._ci
        if ci < len(self._page_rois):
            self._sel = ci
            self._update_grid_btns()
            self._draw_detail()
            self._update_status()

    # ── actions ───────────────────────────────────────────────────────────────

    def _label(self, value):
        self._sess['labels'][self._selected_roi] = value
        save_labels(self._sess)
        self._advance()

    def _advance(self):
        rois = self._page_rois
        if self._sel < len(rois) - 1:
            # move to next ROI on this page
            self._sel += 1
            self._update_grid_btns()
            self._draw_detail()
        elif self._page < self._n_pages - 1:
            # move to next page
            self._page += 1
            self._sel   = 0
            self._draw_grid()
            self._draw_detail()
            self._update_grid_btns()
        elif self._si < len(self._paths) - 1:
            # end of session → auto-load next session
            print(f'  ✓ Session {self._si+1} done — loading next session …')
            self._load_session(self._si + 1)
            self._draw_grid()
            self._draw_detail()
            self._update_grid_btns()
        else:
            self._status.value += '  <b style="color:#27ae60">✓ All sessions complete!</b>'
        self._update_status()

    def _prev_page(self):
        if self._page > 0:
            self._page -= 1; self._sel = 0
            self._draw_grid(); self._draw_detail()
            self._update_grid_btns(); self._update_status()

    def _next_page(self):
        if self._page < self._n_pages - 1:
            self._page += 1; self._sel = 0
            self._draw_grid(); self._draw_detail()
            self._update_grid_btns(); self._update_status()

    def _prev_session(self):
        if self._si > 0:
            self._load_session(self._si - 1)
            self._draw_grid(); self._draw_detail()
            self._update_grid_btns(); self._update_status()
        else:
            print('Already at the first session.')

    def _next_session(self):
        if self._si < len(self._paths) - 1:
            self._load_session(self._si + 1)
            self._draw_grid(); self._draw_detail()
            self._update_grid_btns(); self._update_status()
        else:
            print('Already at the last session.')

In [12]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
BASE_PATH    = '/storage/project/r-fnajafi3-0/shared/2P_Imaging'
MOUSE_FOLDER = 'SA19_LG'
DATES = [                   # list all sessions you want to review; leave empty [] to auto-discover
    #'20251226',
    #'20251228',
    #'20251229',
    #'20251230',
    #'20251231',
    #'20260103',
    #'20260104',
    #'20260105',
    #'20260106',
    '20260107',
    #'20260111',
    #'20260112',
]

PIPELINE_DIR = '/storage/project/r-fnajafi3-0/saminnaji3/Projects/Joystick_Final_Versions/PostProcessing'
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, PIPELINE_DIR)
import config

DISPLAY_SECONDS = 30                          # seconds to show in each trace window
DISPLAY_FRAMES  = int(DISPLAY_SECONDS * config.FS)

## Discover sessions

In [13]:
SESSIONS = discover_sessions()
print(f'Sessions ready ({len(SESSIONS)} total):')
for i, p in enumerate(SESSIONS):
    tag = '✓' if os.path.isfile(os.path.join(p, 'ROI_label.h5')) else '○'
    print(f'  [{i:2d}] {tag}  {os.path.basename(p)}')

Sessions ready (1 total):
  [ 0] ○  SA19_20260107


## Interactive ROI Reviewer

## Launch

In [14]:
if not SESSIONS:
    print('No sessions found. Check BASE_PATH / MOUSE_FOLDER / DATES and run step 1 first.')
else:
    reviewer = ROIReviewer(SESSIONS)

Loading session 1/1: SA19_20260107 …
  1274 ROIs  |  80 pages


HTML(value='')

In [15]:
display(reviewer._btn_good)
display(reviewer._btn_bad)
display(reviewer._btn_skip)

Button(button_style='success', description='✓ Good (G)', layout=Layout(height='32px', width='95px'), style=But…

Button(button_style='danger', description='✗ Bad (B)', layout=Layout(height='32px', width='95px'), style=Butto…

Button(description='→ Skip', layout=Layout(height='32px', width='95px'), style=ButtonStyle())

In [18]:
display(reviewer._btn_good)
display(reviewer._btn_bad)
display(reviewer._btn_skip)

Loading session 3/10: SA19_20251229 …
  498 ROIs  |  32 pages
